# 01 — Análise Exploratória dos Dados (EDA)

Neste notebook vamos explorar os dois datasets baixados do Space-Track:
- **satcat**: catálogo histórico de todos os objetos lançados desde 1957
- **gp_leo**: elementos orbitais atuais de objetos em órbita baixa (LEO)

Objetivo: entender a estrutura dos dados, identificar padrões e preparar o terreno para o modelo de risco.

## 0. Imports e Configurações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# Estilo dos gráficos
sns.set_theme(style='darkgrid', palette='tab10')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 110

RAW_DIR = Path('../data/raw')
OUT_DIR = Path('../outputs')
OUT_DIR.mkdir(exist_ok=True)

print('Imports OK')

## 1. Carregando os Dados

In [ ]:
# Catálogo histórico
satcat = pd.read_csv(RAW_DIR / 'satcat.csv', low_memory=False)
print(f'satcat: {len(satcat):,} linhas | {satcat.shape[1]} colunas')

# Elementos orbitais LEO atuais
gp = pd.read_csv(RAW_DIR / 'gp_leo.csv', low_memory=False)
print(f'gp_leo: {len(gp):,} linhas | {gp.shape[1]} colunas')

## 2. Limpeza Básica do satcat

In [ ]:
# Converter datas
satcat['LAUNCH'] = pd.to_datetime(satcat['LAUNCH'], errors='coerce')
satcat['DECAY']  = pd.to_datetime(satcat['DECAY'],  errors='coerce')

# Criar coluna de ano de lançamento (já existe, mas vamos garantir como int)
satcat['LAUNCH_YEAR'] = satcat['LAUNCH'].dt.year

# Flag: objeto ainda em órbita (CURRENT == 'Y' e sem data de decay)
satcat['EM_ORBITA'] = (satcat['CURRENT'] == 'Y') & satcat['DECAY'].isna()

# Remover linhas sem ano de lançamento
satcat = satcat.dropna(subset=['LAUNCH_YEAR'])
satcat['LAUNCH_YEAR'] = satcat['LAUNCH_YEAR'].astype(int)

print('Tipos de objeto:')
print(satcat['OBJECT_TYPE'].value_counts())
print(f'\nTotal em órbita hoje: {satcat["EM_ORBITA"].sum():,}')

## 3. Crescimento de Objetos ao Longo do Tempo

Quantos objetos foram lançados por ano? Como evoluiu a quantidade acumulada?

In [ ]:
# Lançamentos por ano e por tipo de objeto
por_ano = (
    satcat
    .groupby(['LAUNCH_YEAR', 'OBJECT_TYPE'])
    .size()
    .reset_index(name='COUNT')
)

# Total por ano (todos os tipos)
total_por_ano = satcat.groupby('LAUNCH_YEAR').size().reset_index(name='COUNT')
total_por_ano['ACUMULADO'] = total_por_ano['COUNT'].cumsum()

# Gráfico
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Lançamentos por ano (barras empilhadas por tipo) ---
pivot = por_ano.pivot_table(index='LAUNCH_YEAR', columns='OBJECT_TYPE', values='COUNT', fill_value=0)
pivot.plot(kind='bar', stacked=True, ax=axes[0], width=0.85, colormap='tab10')
axes[0].set_title('Objetos Lançados por Ano e Tipo', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Ano')
axes[0].set_ylabel('Quantidade')
axes[0].xaxis.set_major_locator(mticker.MultipleLocator(10))
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(title='Tipo', loc='upper left')

# --- Total acumulado ---
axes[1].plot(total_por_ano['LAUNCH_YEAR'], total_por_ano['ACUMULADO'], color='steelblue', linewidth=2)
axes[1].fill_between(total_por_ano['LAUNCH_YEAR'], total_por_ano['ACUMULADO'], alpha=0.2, color='steelblue')
axes[1].set_title('Total Acumulado de Objetos no Espaço (desde 1957)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Ano')
axes[1].set_ylabel('Total acumulado')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Anotação de eventos importantes
eventos = {
    1957: 'Sputnik 1',
    2007: 'Fengyun-1C\n(teste ASAT)',
    2009: 'Iridium/Cosmos\nColisão',
    2019: 'Starlink\nInício'
}
for ano, label in eventos.items():
    acum = total_por_ano.loc[total_por_ano['LAUNCH_YEAR'] == ano, 'ACUMULADO']
    if not acum.empty:
        axes[1].annotate(label, xy=(ano, acum.values[0]),
                         xytext=(ano+2, acum.values[0] + 1500),
                         fontsize=7.5, color='darkred',
                         arrowprops=dict(arrowstyle='->', color='darkred', lw=1))

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig01_crescimento_temporal.png', bbox_inches='tight')
plt.show()
print('Gráfico salvo em outputs/fig01_crescimento_temporal.png')

## 4. Composição Atual da Órbita

Do que é feito o lixo orbital hoje?

In [ ]:
# Apenas objetos ainda em órbita
em_orbita = satcat[satcat['EM_ORBITA']]

composicao = em_orbita['OBJECT_TYPE'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Pizza ---
cores = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
axes[0].pie(
    composicao.values,
    labels=composicao.index,
    autopct='%1.1f%%',
    colors=cores,
    startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
axes[0].set_title('Composição de Objetos em Órbita Hoje', fontsize=13, fontweight='bold')

# --- Barra horizontal ---
axes[1].barh(composicao.index, composicao.values, color=cores)
for i, v in enumerate(composicao.values):
    axes[1].text(v + 30, i, f'{v:,}', va='center', fontsize=10)
axes[1].set_xlabel('Quantidade de Objetos')
axes[1].set_title('Objetos em Órbita por Tipo', fontsize=13, fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig02_composicao_orbital.png', bbox_inches='tight')
plt.show()

print('Composição atual:')
for tipo, qtd in composicao.items():
    print(f'  {tipo:20s}: {qtd:,}')
print(f'  {"TOTAL":20s}: {composicao.sum():,}')

## 5. Distribuição por Altitude em LEO

Onde exatamente em LEO estão concentrados os objetos? Quais altitudes são as mais "lotadas"?

In [ ]:
# Altitude média de cada objeto = média entre apogeu e perigeu
gp['ALT_MEDIA'] = (gp['APOAPSIS'] + gp['PERIAPSIS']) / 2

# Filtrar altitudes válidas para LEO (200 a 2000 km)
gp_leo = gp[(gp['ALT_MEDIA'] >= 200) & (gp['ALT_MEDIA'] <= 2000)].copy()

print(f'Objetos com altitude válida em LEO: {len(gp_leo):,}')
print(f'Altitude mínima: {gp_leo["ALT_MEDIA"].min():.0f} km')
print(f'Altitude máxima: {gp_leo["ALT_MEDIA"].max():.0f} km')
print(f'Altitude média:  {gp_leo["ALT_MEDIA"].mean():.0f} km')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Histograma geral ---
axes[0].hist(gp_leo['ALT_MEDIA'], bins=90, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].set_xlabel('Altitude Média (km)')
axes[0].set_ylabel('Número de Objetos')
axes[0].set_title('Distribuição de Objetos por Altitude em LEO', fontsize=13, fontweight='bold')

# Marcas de referência
referencias = {'ISS (~420 km)': 420, 'Starlink (~550 km)': 550, 'Sun-sync (~700 km)': 700}
cores_ref = ['green', 'orange', 'red']
for (label, alt), cor in zip(referencias.items(), cores_ref):
    axes[0].axvline(alt, color=cor, linestyle='--', linewidth=1.5, label=label)
axes[0].legend()

# --- Por tipo de objeto ---
tipos_cores = {'PAYLOAD': '#3498db', 'DEBRIS': '#e74c3c', 'ROCKET BODY': '#f39c12', 'UNKNOWN': '#95a5a6'}
for tipo, cor in tipos_cores.items():
    subset = gp_leo[gp_leo['OBJECT_TYPE'] == tipo]['ALT_MEDIA']
    if len(subset) > 0:
        axes[1].hist(subset, bins=60, alpha=0.6, label=f'{tipo} ({len(subset):,})', color=cor, edgecolor='white', linewidth=0.2)

axes[1].set_xlabel('Altitude Média (km)')
axes[1].set_ylabel('Número de Objetos')
axes[1].set_title('Distribuição por Altitude — Separado por Tipo', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig03_distribuicao_altitude.png', bbox_inches='tight')
plt.show()

## 6. Faixas de Altitude — Densidade por Camada

Vamos dividir LEO em faixas de 100 km e contar objetos em cada faixa. Essa será a base da nossa métrica de risco.

In [ ]:
# Criar faixas de altitude de 100 em 100 km
bins   = list(range(200, 2100, 100))
labels = [f'{b}-{b+100} km' for b in bins[:-1]]

gp_leo['FAIXA_ALT'] = pd.cut(gp_leo['ALT_MEDIA'], bins=bins, labels=labels)

# Contar por faixa e tipo
densidade = (
    gp_leo
    .groupby(['FAIXA_ALT', 'OBJECT_TYPE'], observed=True)
    .size()
    .reset_index(name='COUNT')
)

# Pivot para empilhar
pivot_faixas = densidade.pivot_table(
    index='FAIXA_ALT', columns='OBJECT_TYPE', values='COUNT', fill_value=0
)

# Gráfico de barras horizontais empilhadas
fig, ax = plt.subplots(figsize=(10, 14))

cores_tipo = {'PAYLOAD': '#3498db', 'DEBRIS': '#e74c3c', 'ROCKET BODY': '#f39c12', 'UNKNOWN': '#95a5a6'}
bottom = np.zeros(len(pivot_faixas))

for tipo in pivot_faixas.columns:
    valores = pivot_faixas[tipo].values
    cor = cores_tipo.get(tipo, 'gray')
    ax.barh(pivot_faixas.index.astype(str), valores, left=bottom, label=tipo, color=cor, alpha=0.85)
    bottom += valores

ax.set_xlabel('Número de Objetos', fontsize=11)
ax.set_ylabel('Faixa de Altitude', fontsize=11)
ax.set_title('Densidade de Objetos por Faixa de Altitude em LEO\n(faixas de 100 km)', fontsize=13, fontweight='bold')
ax.legend(title='Tipo de Objeto', loc='lower right')

# Adicionar total no final de cada barra
totais = pivot_faixas.sum(axis=1)
for i, (faixa, total) in enumerate(totais.items()):
    if total > 0:
        ax.text(total + 10, i, f'{int(total):,}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig04_densidade_faixas.png', bbox_inches='tight')
plt.show()

# Top 5 faixas mais lotadas
print('\nTop 10 faixas de altitude mais densas:')
print(totais.sort_values(ascending=False).head(10).to_string())

## 7. Quais países têm mais objetos em LEO?

In [ ]:
top_paises = (
    gp_leo['COUNTRY_CODE']
    .value_counts()
    .head(15)
)

fig, ax = plt.subplots(figsize=(10, 5))
cores = plt.cm.tab20(np.linspace(0, 1, len(top_paises)))
bars = ax.bar(top_paises.index, top_paises.values, color=cores, edgecolor='white')

for bar, val in zip(bars, top_paises.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{val:,}', ha='center', fontsize=9)

ax.set_title('Top 15 Países com Mais Objetos em LEO', fontsize=13, fontweight='bold')
ax.set_xlabel('País')
ax.set_ylabel('Número de Objetos')
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig05_paises.png', bbox_inches='tight')
plt.show()

## 8. Crescimento de Debris ao Longo do Tempo

Debris é o principal fator de risco. Como cresceu ao longo das décadas?

In [ ]:
# Acumulado por tipo ao longo dos anos
tipos_interesse = ['PAYLOAD', 'DEBRIS', 'ROCKET BODY']

fig, ax = plt.subplots(figsize=(13, 5))

cores_linha = {'PAYLOAD': '#3498db', 'DEBRIS': '#e74c3c', 'ROCKET BODY': '#f39c12'}

for tipo in tipos_interesse:
    subset = satcat[satcat['OBJECT_TYPE'] == tipo]
    acum = subset.groupby('LAUNCH_YEAR').size().cumsum().reset_index(name='COUNT')
    ax.plot(acum['LAUNCH_YEAR'], acum['COUNT'],
            label=tipo, linewidth=2.2, color=cores_linha[tipo])

# Eventos de debris
ax.axvline(2007, color='gray', linestyle=':', linewidth=1.2)
ax.text(2007.3, ax.get_ylim()[1]*0.6, 'Fengyun-1C\n(teste ASAT China)', fontsize=8, color='gray')

ax.axvline(2009, color='gray', linestyle=':', linewidth=1.2)
ax.text(2009.3, ax.get_ylim()[1]*0.4, 'Iridium 33 x\nCosmos 2251', fontsize=8, color='gray')

ax.set_title('Crescimento Acumulado por Tipo de Objeto (1957–hoje)', fontsize=13, fontweight='bold')
ax.set_xlabel('Ano')
ax.set_ylabel('Total Acumulado')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(title='Tipo')

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig06_crescimento_por_tipo.png', bbox_inches='tight')
plt.show()

## 9. Resumo da EDA

O que aprendemos com os dados?

In [ ]:
print('=' * 55)
print('  RESUMO DA EDA')
print('=' * 55)

print(f'\n Total de objetos catalogados desde 1957 : {len(satcat):,}')
print(f' Objetos ainda em órbita hoje           : {satcat["EM_ORBITA"].sum():,}')
print(f' Objetos em LEO (dados orbitais)        : {len(gp_leo):,}')

print('\n Composição em órbita:')
for tipo, qtd in satcat[satcat['EM_ORBITA']]['OBJECT_TYPE'].value_counts().items():
    print(f'   {tipo:20s}: {qtd:,}')

print(f'\n Faixa de altitude mais densa em LEO:')
mais_densa = totais.sort_values(ascending=False).index[0]
mais_densa_qtd = int(totais.sort_values(ascending=False).values[0])
print(f'   {mais_densa} com {mais_densa_qtd:,} objetos')

print(f'\n Gráficos salvos em: outputs/')
print('=' * 55)
print('\n Próximo passo: 02_features.ipynb')
print(' → Calcular a métrica de densidade/risco por faixa de altitude')